## Demographics Summary

In [7]:
import pandas as pd

#path to demographics summary file
demo_path = ("../data/demographics_P2.xlsx")

demographics = pd.read_excel(demo_path, keep_default_na = False)

display(demographics.style.hide(axis="index"))

Category,Subcategory,CAT (N=53),Any Blood (N=151),Total (N=1202)
Age,,,,
,Mean,42.700000,43.380000,43.240000
,Median,40,40,40
,Max,82,82,88
,Min,19,18,18
Gender,,,,
,Male,40 (75.5%),114 (75.5%),828 (68.9%)
,Female,12 (22.6%),36 (23.8%),372 (30.9%)
,Unkown,1 (1.9%),1 (0.7%),2 (0.2%)
Race,,,,


## Features Analysis

In [2]:
#path to features file
features_path = ("../data/features_P2_inH_15min.xlsx")

features = pd.read_excel(features_path, index_col = 0, keep_default_na = False)

features.head()

,CAT,Any Blood,HR_Mean,HR_STD,HR_Variance,HR_Min,HR_Max,HR_CoeffOfVar,HR_10pctl,HR_20pctl,...,S2_mode,S1S2ratio_min,S1S2ratio_1stquatile,S1S2ratio_2ndquatile,S1S2ratio_3rdquatile,S1S2ratio_max,S1S2ratio_mode,dnotch_pos_speed_dur_2ndquatile,dnotch_pos_speed_dur_3rdquatile,dnotchpospct
id,,,,,,,,,,,,,,,,,,,,,
P2_0001,NO,NO,79.973,1.938,3.754,74,86,0.925,78,79,...,50,0.045,2.412,2.868,3.512,18.167,3,0.2,0.5,0.585
P2_0002,NO,NO,65.349,2.238,5.008,60,71,0.918,63,63,...,51,0.052,2.03,2.896,4.068,126.546,3,0.222,0.667,0.587
P2_0003,NO,NO,54.646,3.524,12.419,48,64,0.878,51,52,...,57.175,0.034,1.129,2.368,4.878,72.236,1,0,0.333,0.363
P2_0004,NO,NO,100.745,2.19,4.794,97,106,0.963,98,99,...,31,0.038,1.973,2.489,3.29,16.027,3,0.231,0.667,0.653
P2_0005,NO,NO,108.277,1.692,2.862,105,112,0.97,106,107,...,44,0.018,1.437,1.945,2.515,16.08,2,0,0,0.146


### Split into CAT and non CAT cases

In [3]:
CAT_cases = ['P2_0021','P2_0049','P2_0112','P2_0144','P2_0179','P2_0188','P2_0189','P2_0201','P2_0212','P2_0216','P2_0218','P2_0247','P2_0254',
'P2_0266','P2_0290','P2_0303','P2_0309','P2_0313','P2_0314','P2_0442','P2_0450','P2_0471','P2_0484','P2_0519','P2_0610','P2_0617','P2_0670',
'P2_0673','P2_0680','P2_0692','P2_0748','P2_0778','P2_0790','P2_0829','P2_0839','P2_0842','P2_0849','P2_0893','P2_0898','P2_0914','P2_0949',
'P2_0982','P2_1004','P2_1031','P2_1062','P2_1110','P2_1130','P2_1144','P2_1148','P2_1151','P2_1156','P2_1169','P2_1187']

#separating dataframe into CAT and non CAT
CAT_features = features.loc[CAT_cases]

non_CAT_features = features.drop(CAT_cases)

### Create Vital Signs Heatmaps

In [4]:
import plotly.express as px
from scipy.cluster.hierarchy import linkage, leaves_list

#define which vital signs to plot
vital_signs = ["HR", "SPO2p", "SBP", "MBP", "DBP", "RESP", "PP"]

#define ranges for the variables for the heatmap to compare between CAT and non CAT
ranges = {
    "HR": (0,200),
    "SPO2p": (20,100),
    "SBP": (20, 300),
    "MBP": (0, 200),
    "DBP": (0, 180),
    "RESP": (0,90),
    "PP": (0, 150)
}

for vs in vital_signs:
    #define percentile columns to plot
    percentile_cols = [vs+'_10pctl', vs+'_20pctl', vs+'_30pctl', vs+'_40pctl', vs+'_Median', vs+'_60pctl', vs+'_70pctl', vs+'_80pctl', vs+'_90pctl']

    #process CAT features
    cut_df = CAT_features[percentile_cols].copy()

    #remove empty rows
    cut_df[percentile_cols] = cut_df[percentile_cols].apply(
        pd.to_numeric,
        errors='coerce'
    )
    
    cut_df = cut_df.dropna()

    #apply sorting
    df_sorted = cut_df.sort_values(vs+'_Median')

    #configure plot
    fig = px.imshow(
        df_sorted,
        aspect="auto",
        color_continuous_scale="Viridis",
        labels={
            "x": "Percentile",
            "y": "Case",
            "color": "Value"
        },
        title="CAT " + vs + " Heatmap",
        zmin=ranges[vs][0],
        zmax=ranges[vs][1]
    )

    #remove labels
    fig.update_yaxes(showticklabels=False)

    #save to interactive html
    fig.write_html(
        "CAT " + vs + " Heatmap.html",
        include_plotlyjs="cdn",
        auto_open=True
    )

    #process non CAT features
    cut_df = non_CAT_features[percentile_cols].copy()

    #remove empty rows
    cut_df[percentile_cols] = cut_df[percentile_cols].apply(
        pd.to_numeric,
        errors='coerce'
    )
    
    cut_df = cut_df.dropna()

    #apply sorting
    df_sorted = cut_df.sort_values(vs+'_Median')

    #configure plot
    fig = px.imshow(
        df_sorted,
        aspect="auto",
        color_continuous_scale="Viridis",
        labels={
            "x": "Percentile",
            "y": "Case",
            "color": "Value"
        },
        title="non CAT " + vs + " Heatmap",
        zmin=ranges[vs][0],
        zmax=ranges[vs][1]
    )

    #remove labels
    fig.update_yaxes(showticklabels=False)

    #save to interactive html
    fig.write_html(
        "non CAT " + vs + " Heatmap.html",
        include_plotlyjs="cdn",
        auto_open=True
    )